In [1]:
#imports
import cv2
# import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image, ImageOps
from dataclasses import dataclass, field
from typing import Optional, List
import io
import tempfile

from paddleocr import PaddleOCR
from pdf2image import convert_from_path
import pymupdf4llm

# %matplotlib inline

/home/rishicarter/git/AI-Space/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Constants
MAX_LONG_SIDE: int = 800    # Resize longest dimension to this (px). Reduces RAM ~2-4x.
                             # Safe floor for PaddleOCR legibility: 640. Max recommended: 1280.
MIN_LONG_SIDE: int = 640    # Never downsample below this (avoids blurring small text).
CONF_THRESHOLD: float = 0.5 # Discard detections below this confidence score.

_ocr_instance = None

In [3]:
ocr = PaddleOCR(
    use_doc_orientation_classify=True,
    use_doc_unwarping=True,
    use_textline_orientation=True,
    enable_mkldnn=False
)

/home/rishicarter/git/AI-Space/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/rishicarter/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/rishicarter/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/rishicarter/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creatin

In [4]:
ocr.predict??

Signature:
ocr.predict(
    input,
    *,
    use_doc_orientation_classify=None,
    use_doc_unwarping=None,
    use_textline_orientation=None,
    text_det_limit_side_len=None,
    text_det_limit_type=None,
    text_det_thresh=None,
    text_det_box_thresh=None,
    text_det_unclip_ratio=None,
    text_rec_score_thresh=None,
    return_word_box=None,
)
Docstring: <no docstring>
Source:   
    def predict(
        self,
        input,
        *,
        use_doc_orientation_classify=None,
        use_doc_unwarping=None,
        use_textline_orientation=None,
        text_det_limit_side_len=None,
        text_det_limit_type=None,
        text_det_thresh=None,
        text_det_box_thresh=None,
        text_det_unclip_ratio=None,
        text_rec_score_thresh=None,
        return_word_box=None,
    ):
        return list(
            self.predict_iter(
                input,
                use_doc_orientation_classify=use_doc_orientation_classify,
                use_doc_unwarping=use_doc